# Fine-tuning QLoRA da Fase 3

Este notebook treina o adaptador Qwen 2.5 0.5B usando somente os exemplos médicos sintéticos do projeto. Antes, gere o pacote ZIP no computador com python scripts/create_colab_bundle.py e selecione GPU em **Ambiente de execução → Alterar tipo de ambiente de execução**.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os

uploaded = files.upload()
archive_name = next(name for name in uploaded if name.endswith('.zip'))
workdir = Path('/content/phase3')
workdir.mkdir(parents=True, exist_ok=True)
with ZipFile(archive_name) as archive:
    archive.extractall(workdir)
os.chdir(workdir / 'code')
print('Projeto extraído em', Path.cwd())

In [ ]:
!nvidia-smi
!python -m pip install --upgrade -r requirements-training.txt
!python -m pip check

import torch, transformers, peft, datasets, accelerate, bitsandbytes, huggingface_hub, fsspec
print({
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'peft': peft.__version__,
    'datasets': datasets.__version__,
    'accelerate': accelerate.__version__,
    'bitsandbytes': bitsandbytes.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'fsspec': fsspec.__version__,
    'cuda_available': torch.cuda.is_available(),
})
assert torch.cuda.is_available(), 'Ative a GPU em Ambiente de execu\u00e7\u00e3o > Alterar tipo de ambiente de execu\u00e7\u00e3o.'

In [ ]:
!python scripts/generate_instruction_data.py
!python scripts/ingest_corpus.py
!python scripts/run_evaluation.py
!python scripts/train_lora.py --run --qlora

In [ ]:
!python scripts/run_generation_evaluation.py --limit 90

In [ ]:
from google.colab import files
import shutil

adapter = Path('outputs/lora_adapter')
archive = shutil.make_archive('/content/fase3_lora_adapter', 'zip', root_dir=adapter)
files.download(archive)